# 02 编码器模块 (core.encoders)

覆盖 WOE / Target / Count / OneHot / Ordinal / Quantile / CatBoost / GBM 编码器。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import hscredit as hsc
from hscredit import germancredit, init_setting
from hscredit import (
    WOEEncoder, TargetEncoder, CountEncoder, OneHotEncoder,
    OrdinalEncoder, QuantileEncoder,
)
from sklearn.model_selection import train_test_split
init_setting()
df = germancredit()
target = 'creditability'
feats = ['purpose', 'credit.history', 'duration.in.month', 'credit.amount']
X = df[feats]; y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
print(X_tr.shape, X_te.shape)

## 1. WOE 编码

In [ ]:
woe = WOEEncoder(max_n_bins=5)
woe.fit(X_tr, y_tr)
X_woe = woe.transform(X_te)
print('WOE编码结果:')
print(X_woe.head(3))
print('\nWOE映射表:')
for col, tbl in woe.woe_tables_.items():
    print(f'\n--- {col} ---')
    print(tbl[['分箱标签','分档WOE值','分档IV值']].head())

## 2. Target 编码

In [ ]:
te = TargetEncoder(smoothing=5.0)
te.fit(X_tr, y_tr)
X_te_enc = te.transform(X_te)
print(X_te_enc.head(3))

## 3. Count 编码

In [ ]:
ce = CountEncoder()
ce.fit(X_tr, y_tr)
print(ce.transform(X_te).head(3))

## 4. OneHot 编码

In [ ]:
cat_feats = ['purpose', 'credit.history']
ohe = OneHotEncoder(max_categories=5)
ohe.fit(X_tr[cat_feats], y_tr)
print(ohe.transform(X_te[cat_feats]).head(3))

## 5. Ordinal 编码

In [ ]:
oe = OrdinalEncoder()
oe.fit(X_tr[cat_feats], y_tr)
print(oe.transform(X_te[cat_feats]).head(3))

## 6. Quantile 编码

In [ ]:
qe = QuantileEncoder(n_quantiles=10)
qe.fit(X_tr, y_tr)
print(qe.transform(X_te).head(3))

## 7. Pipeline 中使用编码器

In [ ]:
from hscredit import Pipeline, OptimalBinning
from sklearn.linear_model import LogisticRegression
pipe = Pipeline([
    ('woe', WOEEncoder(max_n_bins=5)),
    ('lr', LogisticRegression(max_iter=1000)),
])
pipe.fit(X_tr, y_tr)
print('Pipeline AUC:', pipe.score(X_te, y_te))